# Hindustani Raga Classifier — trained from scratch on the TRF dataset

**Why this notebook exists**: this project's fused similarity vector used to include a `raga_probability` feature that turned out to be fake. Research into real alternatives found DeepSRGM's pretrained checkpoint, but it only classifies 10 **Carnatic** ragas — wrong tradition for a Bollywood/ghazal/sufi catalog. The [Thaat and Raga Forest (TRF) dataset](https://www.kaggle.com/datasets/suryamajumder/thaat-and-raga-forest-trf-dataset) is **Hindustani** (10 thaats, 61 ragas — Bhairavi, Yaman, Khamaj, Malkauns, Desh...) and explicitly includes movie songs, but has **no pretrained weights anywhere**. So: train one ourselves.

**On the paper's own disclosed accuracy (93–100%)**: don't expect to reproduce it, and don't treat it as a target. The paper's own dataset description says raw recordings were "augmented" into many 30s/60s clips with no mention of keeping one recording's clips together across train/test — a classic setup for clip-leakage inflating accuracy to look far better than real generalization. This notebook splits by **recording**, never by clip, specifically to avoid that. Expect a real, probably much lower, number — and trust it more for being real.

**Real timing, not a guess**: ~19GB / 1103 recordings / an estimated 250-270 hours of total audio. Decoding a fresh random crop from disk on every training step, every epoch, would very plausibly exceed Kaggle's 12-hour session limit before training even gets going. So this notebook **precomputes every recording's log-mel spectrogram segments ONCE up front, caches them to disk, and is resumable** — if precompute doesn't finish in one session, save the cache as a Kaggle Dataset, attach it as `EXISTING_CACHE_INPUT` on your next run, and it picks up where it left off (skips any file already cached). Training itself also checkpoints every epoch and can resume the same way via `EXISTING_CHECKPOINT_INPUT`. The precompute cell prints progress + an extrapolated ETA after the first few files so you get an honest time estimate early, not a guess from me.

**Yes, you get real results at the end** — per-segment AND per-file majority-vote held-out test accuracy (majority-vote is what matters: classifying a whole song, not one clip), plus a per-class breakdown, all written to `results.json`.

**Before running:**
1. Add Data → search `suryamajumder/thaat-and-raga-forest-trf-dataset` → Add (~19GB, mounted read-only, no download on your end).
2. Notebook Settings → Accelerator → GPU T4 x2.
3. If resuming a prior session: also Add Data your previously-saved cache/checkpoint dataset(s) and set `EXISTING_CACHE_INPUT` / `EXISTING_CHECKPOINT_INPUT` below.
4. **After the session ends (whether or not training finished)**: File → Save Version, so the cache and/or checkpoint persist for next time.

In [ ]:
# --- Config ---
SAMPLE_RATE = 22050
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
SEGMENT_SECONDS = 30
MIN_SEGMENT_SECONDS = 15   # drop a trailing chunk shorter than this
MAX_SEGMENTS_PER_FILE = 15 # cap so one 40min recording doesn't dominate a 4min one

PRECOMPUTE_WORKERS = 4     # parallel processes for the one-time decode+spectrogram pass

BATCH_SIZE = 32
EPOCHS = 25
LR = 1e-3
EARLY_STOP_PATIENCE = 6
MAX_TRAIN_SECONDS = 8 * 3600

SEED = 42
NUM_WORKERS = 2

SEGMENTS_CACHE_DIR = "/kaggle/working/segment_cache"
CHECKPOINT_PATH = "/kaggle/working/last_checkpoint.pt"
BEST_MODEL_PATH = "/kaggle/working/raga_model_best.pt"

# Set these to resume across sessions — point at a dataset you Saved Version'd
# from a previous run and re-attached via Add Data.
EXISTING_CACHE_INPUT = None       # e.g. "/kaggle/input/raga-training-checkpoint/segment_cache"
EXISTING_CHECKPOINT_INPUT = None  # e.g. "/kaggle/input/raga-training-checkpoint/last_checkpoint.pt"

In [ ]:
import subprocess, time, os

def run(cmd, timeout, label=None):
    label = label or cmd
    print(f"--- RUNNING ({timeout}s timeout): {label}")
    t0 = time.time()
    try:
        result = subprocess.run(cmd, shell=True, timeout=timeout,
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except subprocess.TimeoutExpired as e:
        print(e.stdout or "")
        raise RuntimeError(f"TIMED OUT after {time.time()-t0:.0f}s: {label}")
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"FAILED (exit {result.returncode}): {label}")
    print(f"--- OK ({time.time()-t0:.0f}s): {label}")
    return result

run("apt-get -qq install -y ffmpeg", timeout=120, label="apt-get ffmpeg")
os.makedirs(SEGMENTS_CACHE_DIR, exist_ok=True)

In [ ]:
# --- Bring in a prior session's cache/checkpoint, if resuming ---
import shutil

if EXISTING_CACHE_INPUT:
    if not os.path.isdir(EXISTING_CACHE_INPUT):
        raise RuntimeError(f"EXISTING_CACHE_INPUT={EXISTING_CACHE_INPUT!r} not found. Check it's attached via Add Data.")
    shutil.copytree(EXISTING_CACHE_INPUT, SEGMENTS_CACHE_DIR, dirs_exist_ok=True)
    print(f"Copied in {len(os.listdir(SEGMENTS_CACHE_DIR))} cached files from a prior session.")
else:
    print("No existing cache attached — starting precompute from scratch.")

if EXISTING_CHECKPOINT_INPUT:
    if not os.path.isfile(EXISTING_CHECKPOINT_INPUT):
        raise RuntimeError(f"EXISTING_CHECKPOINT_INPUT={EXISTING_CHECKPOINT_INPUT!r} not found.")
    shutil.copy(EXISTING_CHECKPOINT_INPUT, CHECKPOINT_PATH)
    print("Copied in a prior training checkpoint — will resume from it.")
else:
    print("No existing checkpoint attached — training will start from epoch 1.")

In [ ]:
# --- Locate the dataset and enumerate every recording, labeled by (thaat, raga) ---
import glob
from collections import defaultdict

candidates = [c for c in glob.glob("/kaggle/input/**/Thaat and Raga Forest*", recursive=True) if os.path.isdir(c)]
if not candidates:
    raise RuntimeError("TRF dataset not found under /kaggle/input. Add Data > 'suryamajumder/thaat-and-raga-forest-trf-dataset'.")
DATASET_ROOT = candidates[0]
print("Dataset root:", DATASET_ROOT)

files_by_raga = defaultdict(list)
for path in glob.glob(os.path.join(DATASET_ROOT, "*", "*", "*.mp3")):
    parts = path.split(os.sep)
    files_by_raga[(parts[-3], parts[-2])].append(path)

ragas = sorted(files_by_raga.keys(), key=lambda k: k[1])
raga2idx = {raga_key: i for i, raga_key in enumerate(ragas)}
idx2raga = {i: {"thaat": t, "raga": r} for (t, r), i in raga2idx.items()}
print(f"{len(ragas)} ragas, {sum(len(v) for v in files_by_raga.values())} recordings total")

In [ ]:
# --- File-level train/val/test split (never split one recording's segments across sets) ---
import random
random.seed(SEED)

train_files, val_files, test_files = [], [], []
for raga_key, fs in files_by_raga.items():
    fs = sorted(fs)
    random.shuffle(fs)
    label = raga2idx[raga_key]
    if len(fs) >= 3:
        test_files.append((fs[0], label))
        val_files.append((fs[1], label))
        train_files.extend((f, label) for f in fs[2:])
    else:
        train_files.extend((f, label) for f in fs)

all_files = train_files + val_files + test_files  # precompute covers all of them
print(f"train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")

In [ ]:
# --- Precompute: decode each recording ONCE, cache its log-mel segments to disk.
# Resumable (skips files already cached) and parallelized across CPU cores —
# this is the expensive step; training itself is fast once this is done. ---
import numpy as np
import librosa
import hashlib
from concurrent.futures import ProcessPoolExecutor, as_completed

def _cache_path(path):
    h = hashlib.md5(path.encode()).hexdigest()
    return os.path.join(SEGMENTS_CACHE_DIR, f"{h}.npy")

def _precompute_one(args):
    path, sr, n_mels, n_fft, hop_length, seg_secs, min_secs, max_segs, out_path = args
    if os.path.exists(out_path):
        return path, "cached", None
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)  # ONE full sequential decode
        seg_len = int(seg_secs * sr)
        min_len = int(min_secs * sr)
        starts = range(0, max(1, len(y) - min_len + 1), seg_len)
        segments = []
        for s in starts:
            chunk = y[s:s + seg_len]
            if len(chunk) < min_len:
                continue
            if len(chunk) < seg_len:
                chunk = np.pad(chunk, (0, seg_len - len(chunk)))
            mel = librosa.feature.melspectrogram(y=chunk, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length)
            segments.append(librosa.power_to_db(mel, ref=np.max).astype(np.float32))
            if len(segments) >= max_segs:
                break
        if not segments:
            return path, "empty", None
        np.save(out_path, np.stack(segments))
        return path, "done", None
    except Exception as e:
        return path, "error", str(e)

tasks = [(p, SAMPLE_RATE, N_MELS, N_FFT, HOP_LENGTH, SEGMENT_SECONDS, MIN_SEGMENT_SECONDS,
          MAX_SEGMENTS_PER_FILE, _cache_path(p)) for p, _ in all_files]
already_cached = sum(1 for t in tasks if os.path.exists(t[-1]))
print(f"{already_cached}/{len(tasks)} already cached (resumed) — {len(tasks) - already_cached} left to process")

t0 = time.time()
done, errors = 0, []
with ProcessPoolExecutor(max_workers=PRECOMPUTE_WORKERS) as ex:
    futures = [ex.submit(_precompute_one, t) for t in tasks]
    for i, fut in enumerate(as_completed(futures), 1):
        path, status, err = fut.result()
        if status == "error":
            errors.append((path, err))
            print(f"  ERROR on {path}: {err}")
        done += 1
        if done % 20 == 0 or done == len(tasks):
            elapsed = time.time() - t0
            newly_done = done - already_cached if done > already_cached else done
            rate = elapsed / max(1, done - already_cached) if done > already_cached else None
            remaining = len(tasks) - done
            eta = f"{rate * remaining / 60:.1f}min" if rate else "n/a (still warming up)"
            print(f"  {done}/{len(tasks)} processed, {elapsed/60:.1f}min elapsed, ETA for rest: {eta}")

print(f"Precompute finished: {done} processed, {len(errors)} errors.")
if errors:
    print("Files with errors (will be skipped below):", [e[0] for e in errors])

In [ ]:
# --- Dataset: loads a file's precomputed segment array and returns one at random
# (train) or all of them (val/test, for majority voting). Fast — just disk I/O
# of a small cached array, no audio decoding at train time. ---
import torch
from torch.utils.data import Dataset

def _load_cached(path):
    cache_path = _cache_path(path)
    if not os.path.exists(cache_path):
        return None
    return np.load(cache_path)

class CachedSegmentDataset(Dataset):
    """One random cached segment per file per access (fresh pick each epoch)."""
    def __init__(self, file_label_pairs):
        self.files = [(p, l) for p, l in file_label_pairs if os.path.exists(_cache_path(p))]
        skipped = len(file_label_pairs) - len(self.files)
        if skipped:
            print(f"  (skipping {skipped} files with no cache — precompute errors or still pending)")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx]
        segs = _load_cached(path)
        seg = segs[random.randrange(len(segs))]
        return torch.from_numpy(seg).unsqueeze(0), label

In [ ]:
import torch.nn as nn

class RagaCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True), nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(block(1, 32), block(32, 64), block(64, 128), block(128, 256))
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, 128),
            nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type != "cuda":
    raise RuntimeError("No GPU detected. Check Notebook Settings > Accelerator.")

In [ ]:
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight

train_labels = np.array([label for _, label in train_files])
class_weights_arr = compute_class_weight("balanced", classes=np.arange(len(ragas)), y=train_labels)
class_weights = torch.tensor(class_weights_arr, dtype=torch.float32).to(device)

train_loader = DataLoader(CachedSegmentDataset(train_files), batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, drop_last=True)
val_loader = DataLoader(CachedSegmentDataset(val_files), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS)

model = RagaCNN(num_classes=len(ragas)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

start_epoch = 1
best_val_acc = 0.0
epochs_no_improve = 0
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] + 1
    best_val_acc = ckpt["best_val_acc"]
    epochs_no_improve = ckpt["epochs_no_improve"]
    print(f"Resumed from checkpoint: starting at epoch {start_epoch}, best_val_acc so far={best_val_acc:.3f}")
else:
    print("No checkpoint found — starting fresh at epoch 1.")

print(f"train batches/epoch: {len(train_loader)}  val batches: {len(val_loader)}")

In [ ]:
# --- Training loop — checkpoints EVERY epoch (not just on improvement), so a
# killed session loses at most one epoch's progress, not the whole run. ---
t_start = time.time()

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
        train_correct += (out.argmax(1) == y).sum().item()
        train_total += x.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += x.size(0)

    train_acc = train_correct / max(1, train_total)
    val_acc = val_correct / max(1, val_total)
    scheduler.step(val_acc)
    elapsed = time.time() - t_start
    print(f"epoch {epoch:2d}/{EPOCHS}  train_loss={train_loss/train_total:.4f}  "
          f"train_acc={train_acc:.3f}  val_acc={val_acc:.3f}  elapsed={elapsed/60:.1f}min")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> new best (val_acc={val_acc:.3f}), best-model checkpoint saved")
    else:
        epochs_no_improve += 1

    torch.save({
        "epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(), "best_val_acc": best_val_acc,
        "epochs_no_improve": epochs_no_improve,
    }, CHECKPOINT_PATH)

    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"No val improvement for {EARLY_STOP_PATIENCE} epochs — stopping early.")
        break
    if elapsed > MAX_TRAIN_SECONDS:
        print(f"Hit MAX_TRAIN_SECONDS ({MAX_TRAIN_SECONDS}s) safety limit — stopping. "
              f"Save Version now and resume via EXISTING_CHECKPOINT_INPUT next session.")
        break

print(f"Best val accuracy: {best_val_acc:.3f}")

In [ ]:
# --- Held-out TEST evaluation: per-segment AND per-file majority-vote accuracy,
# using the already-cached test segments (no re-decoding needed). ---
if os.path.exists(BEST_MODEL_PATH):
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)
seg_correct, seg_total = 0, 0
file_correct, file_total = 0, 0

with torch.no_grad():
    for path, label in test_files:
        segs = _load_cached(path)
        if segs is None:
            continue
        x = torch.from_numpy(segs).unsqueeze(1).to(device)
        preds = model(x).argmax(1).cpu().numpy()
        seg_correct += int((preds == label).sum())
        seg_total += len(preds)
        majority = int(np.bincount(preds).argmax())
        file_total += 1
        per_class_total[label] += 1
        if majority == label:
            file_correct += 1
            per_class_correct[label] += 1

print(f"Per-segment test accuracy: {seg_correct/max(1,seg_total):.3f}  ({seg_correct}/{seg_total})")
print(f"Per-file majority-vote test accuracy: {file_correct/max(1,file_total):.3f}  ({file_correct}/{file_total})")
print()
print("Per-class (majority-vote) breakdown:")
for label in sorted(per_class_total):
    print(f"  {idx2raga[label]['raga']:35s} {per_class_correct[label]}/{per_class_total[label]}")

In [ ]:
import json

with open("/kaggle/working/raga_classes.json", "w") as f:
    json.dump(idx2raga, f, indent=2, ensure_ascii=False)
with open("/kaggle/working/preprocessing_config.json", "w") as f:
    json.dump({"sample_rate": SAMPLE_RATE, "n_mels": N_MELS, "n_fft": N_FFT,
               "hop_length": HOP_LENGTH, "segment_seconds": SEGMENT_SECONDS}, f, indent=2)
with open("/kaggle/working/results.json", "w") as f:
    json.dump({
        "best_val_acc": best_val_acc,
        "test_segment_acc": seg_correct / max(1, seg_total),
        "test_majority_vote_acc": file_correct / max(1, file_total),
        "num_ragas": len(ragas),
        "epochs_completed": epoch,
    }, f, indent=2)

print("Saved: raga_model_best.pt, last_checkpoint.pt, segment_cache/, raga_classes.json, preprocessing_config.json, results.json")
print("Click 'Save Version' now — whether or not training finished — to persist the cache and checkpoint for a resumed run.")